In [1]:
# Imports.

from concurrent.futures import ThreadPoolExecutor
import gc
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor, set_seed
from trl import SFTConfig, SFTTrainer

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

/home/tdnguyen/miniforge3/envs/cxr-vlm-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constants.

DATA_CSV = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")
OUTPUT_DIR = Path("artifacts/lora_sft")
MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"
MODEL_DTYPE = torch.bfloat16
RANDOM_STATE = 42

TARGET_LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
PROMPT_ORDERS = ["image_first", "text_first"]
PER_DEVICE_TRAIN_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
IMAGE_LOAD_NUM_WORKERS = 4
IMAGE_PROCESS_BATCH_SIZE = 24
DATALOADER_NUM_WORKERS = 4
DECODER_LINEAR_NAMES = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(RANDOM_STATE)


In [3]:
# Load training studies.

df = pd.read_csv(DATA_CSV)
train_df = df[df["probe_split"] == "train"].reset_index(drop=True)
print("train studies", len(train_df))


train studies 20000


In [4]:
# Processor, prompt format, and cached image tensors.

processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def prompt_text(prompt_order, label):
    question = f"Question: Is there {label.lower()} in this image? Answer yes or no."
    image_item = {"type": "image"}
    if prompt_order == "image_first":
        content = [image_item, {"type": "text", "text": f"\n{question}\nAnswer: "}]
    else:
        content = [{"type": "text", "text": f"{question}\n"}, image_item, {"type": "text", "text": "\nAnswer: "}]
    text = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=False, tokenize=False)
    if isinstance(text, list):
        text = text[0]
    return text.replace(processor.boi_token, processor.full_image_sequence)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

with ThreadPoolExecutor(max_workers=IMAGE_LOAD_NUM_WORKERS) as pool:
    images = list(tqdm(pool.map(load_rgb, train_df["image_path"]), total=len(train_df), desc="Load images"))

pixel_chunks = []
for start in tqdm(range(0, len(images), IMAGE_PROCESS_BATCH_SIZE), desc="Process images"):
    end = min(start + IMAGE_PROCESS_BATCH_SIZE, len(images))
    pixel_inputs = processor.image_processor(images=images[start:end], return_tensors="pt", do_pan_and_scan=False)
    pixel_chunks.append(pixel_inputs["pixel_values"].to(dtype=MODEL_DTYPE, device="cpu"))

cached_pixel_values = torch.cat(pixel_chunks, dim=0)
del images, pixel_chunks
print("cached_pixel_values", tuple(cached_pixel_values.shape), cached_pixel_values.dtype)


Process images: 100%|██████████| 834/834 [16:22<00:00,  1.18s/it]


cached_pixel_values (20000, 3, 896, 896) torch.bfloat16


In [5]:
# Answer-token-only collator.

def collate_text(examples):
    input_rows = []
    label_rows = []
    for example in examples:
        prompt_ids = processor.tokenizer(prompt_text(example["prompt_order"], example["finding"])).input_ids
        answer_ids = processor.tokenizer(example["answer"], add_special_tokens=False).input_ids
        input_rows.append(prompt_ids + answer_ids)
        label_rows.append([-100] * len(prompt_ids) + answer_ids)

    max_len = max(len(row) for row in input_rows)
    input_ids = torch.full((len(examples), max_len), processor.tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)

    for i, (input_row, label_row) in enumerate(zip(input_rows, label_rows)):
        input_ids[i, :len(input_row)] = torch.tensor(input_row)
        attention_mask[i, :len(input_row)] = 1
        labels[i, :len(label_row)] = torch.tensor(label_row)

    batch = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
    if hasattr(processor, "create_mm_token_type_ids"):
        token_type_ids = processor.create_mm_token_type_ids(input_ids)
        batch["token_type_ids"] = token_type_ids if torch.is_tensor(token_type_ids) else torch.tensor(token_type_ids)
    return batch


def collate_batch(examples):
    batch = collate_text(examples)
    batch["pixel_values"] = cached_pixel_values[[int(example["image_idx"]) for example in examples]]
    return batch

sample_examples = []
for image_idx, row in train_df.head(1).iterrows():
    for label in TARGET_LABELS[:2]:
        sample_examples.append({
            "image_idx": image_idx,
            "prompt_order": "image_first",
            "finding": label,
            "answer": "yes" if row[label] == 1 else "no",
        })

sample_batch = collate_batch(sample_examples)
print("yes ids", processor.tokenizer("yes", add_special_tokens=False).input_ids)
print("no ids", processor.tokenizer("no", add_special_tokens=False).input_ids)
print("supervised tokens", sample_batch["labels"].ne(-100).sum(dim=1).tolist())
del sample_batch, sample_examples


yes ids [4443]
no ids [1904]
supervised tokens [1, 1]


In [6]:
# Train one LoRA adapter per prompt order.

for prompt_order in PROMPT_ORDERS:
    rows = []
    for image_idx, row in train_df.iterrows():
        for label in TARGET_LABELS:
            rows.append({
                "image_idx": image_idx,
                "study_id": row["study_id"],
                "prompt_order": prompt_order,
                "finding": label,
                "answer": "yes" if row[label] == 1 else "no",
            })
    train_dataset = Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)
    adapter_dir = OUTPUT_DIR / f"{prompt_order}_adapter"

    print(prompt_order, "examples", len(train_dataset))
    model = AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL_ID, dtype=MODEL_DTYPE)
    model.config.use_cache = False

    target_modules = [
        name for name, module in model.named_modules()
        if isinstance(module, torch.nn.Linear)
        and "language_model" in name
        and "vision_tower" not in name
        and "multi_modal_projector" not in name
        and name.split(".")[-1] in DECODER_LINEAR_NAMES
    ]
    print("LoRA target modules", len(target_modules))
    print(target_modules[:5])

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(adapter_dir),
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            learning_rate=LEARNING_RATE,
            bf16=True,
            max_length=None,
            packing=False,
            warmup_ratio=0.03,
            lr_scheduler_type="cosine",
            max_grad_norm=1.0,
            logging_steps=20,
            save_strategy="epoch",
            save_total_limit=1,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=DATALOADER_NUM_WORKERS,
            dataset_kwargs={"skip_prepare_dataset": True},
            gradient_checkpointing=False,
            seed=RANDOM_STATE,
        ),
        train_dataset=train_dataset,
        data_collator=collate_batch,
        processing_class=processor,
        peft_config=LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=target_modules,
        ),
    )

    trainer.model.print_trainable_parameters()
    trainer.train()
    trainer.save_model(str(adapter_dir))
    processor.save_pretrained(str(adapter_dir))
    pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR / f"{prompt_order}_train_log.csv", index=False)

    del trainer, model, train_dataset
    gc.collect()
    torch.cuda.empty_cache()


image_first examples 100000


Loading weights: 100%|██████████| 883/883 [00:00<00:00, 989.41it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


LoRA target modules 238
['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj', 'model.language_model.layers.0.self_attn.o_proj', 'model.language_model.layers.0.mlp.gate_proj']


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


trainable params: 29,802,496 || all params: 4,329,881,968 || trainable%: 0.6883


...

Loading weights: 100%|██████████| 883/883 [00:00<00:00, 1721.87it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


LoRA target modules 238
['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj', 'model.language_model.layers.0.self_attn.o_proj', 'model.language_model.layers.0.mlp.gate_proj']


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


trainable params: 29,802,496 || all params: 4,329,881,968 || trainable%: 0.6883


Step,Training Loss
20,5.632695
40,0.586216
60,0.563466
80,0.565002
100,0.548580
120,0.522786
140,0.536760
160,0.532841
180,0.506774
200,0.524838
